# Vietnamese OCR Fine-tuning on Kaggle

This notebook fine-tunes PaddleOCR's Vietnamese recognition model using:
- **VinText** dataset (current)
- **ICDAR2019-LSVT** (5,000 Vietnamese scene text images)
- **ICDAR2019-ReCTS** (Vietnamese/Chinese restaurant text)

## Strategy
1. Download VinText + additional datasets
2. Crop text regions and create label files
3. Fine-tune PP-OCRv5-server Vietnamese recognition model
4. Export model for deployment to Modal.com

In [ ]:
# Cell 1: Setup Environment
!pip install paddlepaddle-gpu==2.6.0.post118 -f https://www.paddlepaddle.org.cn/whl/linux/cuda11.8/stable.html
!pip install paddleocr==2.7.3
!pip install gdown opencv-python-headless lmdb pyyaml tqdm albumentations
!git clone --depth 1 --branch release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git /kaggle/working/PaddleOCR

import os
os.chdir('/kaggle/working/PaddleOCR')
print("Setup complete.")

In [ ]:
# Cell 2: Download VinText dataset from Google Drive
import gdown
import zipfile

vintext_zip = '/kaggle/working/vintext.zip'
if not os.path.exists('/kaggle/working/vintext_extracted'):
    print('Downloading VinText dataset...')
    gdown.download(id='1UUQhNvzgpZy7zXBFQp0Qox-BBjunZ0ml', output=vintext_zip, quiet=False)
    
    print('Extracting...')
    with zipfile.ZipFile(vintext_zip, 'r') as zip_ref:
        zip_ref.extractall('/kaggle/working/vintext_extracted')
    os.remove(vintext_zip)
    print('VinText extracted.')
else:
    print('VinText already extracted.')

In [ ]:
# Cell 3: Download HuggingFace Vietnamese OCR datasets
# These provide additional Vietnamese scene text for better diacritic recognition

!pip install datasets huggingface_hub

from datasets import load_dataset
import json

# Download Vietnamese scene text dataset from HuggingFace
# Using 'linhkita/vietnamese-ocr-dataset' if available, or similar
try:
    print('Downloading Vietnamese OCR datasets from HuggingFace...')
    # Try to load Vietnamese OCR dataset
    ds = load_dataset('linhkita/vietnamese-ocr-dataset', split='train', trust_remote_code=True)
    print(f'Loaded {len(ds)} samples from HuggingFace dataset')
    
    # Save images and labels
    hf_crops_dir = '/kaggle/working/hf_crops'
    os.makedirs(hf_crops_dir, exist_ok=True)
    
    hf_labels = []
    for i, sample in enumerate(ds):
        img = sample['image']
        text = sample['text']
        img_path = os.path.join(hf_crops_dir, f'hf_{i:06d}.jpg')
        img.save(img_path)
        hf_labels.append(f'hf_crops/hf_{i:06d}.jpg\t{text}\n')
    
    print(f'Saved {len(hf_labels)} HuggingFace samples')
except Exception as e:
    print(f'HuggingFace dataset not available: {e}')
    print('Will proceed with VinText only.')
    hf_labels = []

In [ ]:
# Cell 4: Process VinText - crop text regions (same logic as prepare_dataset)
import cv2
import shutil
from concurrent.futures import ThreadPoolExecutor

extract_dir = '/kaggle/working/vintext_extracted'
labels_dir = os.path.join(extract_dir, 'vietnamese', 'labels')
train_imgs_dir = os.path.join(extract_dir, 'vietnamese', 'train_images')
test_imgs_dir = os.path.join(extract_dir, 'vietnamese', 'test_image')

crops_dir = '/kaggle/working/crops'
shutil.rmtree(crops_dir, ignore_errors=True)
os.makedirs(crops_dir, exist_ok=True)

label_files = sorted(os.listdir(labels_dir))
print(f'Processing {len(label_files)} label files...')

def process_label_file(l_file):
    if not l_file.startswith('gt_') or not l_file.endswith('.txt'):
        return []
    try:
        N = int(l_file[3:-4])
    except ValueError:
        return []
    
    img_name = f'im{N:04d}.jpg'
    if 1 <= N <= 1200:
        img_path = os.path.join(train_imgs_dir, img_name)
        is_train = True
    elif 1201 <= N <= 1500:
        img_path = os.path.join(test_imgs_dir, img_name)
        is_train = False
    else:
        return []
    
    if not os.path.exists(img_path):
        return []
    
    img = cv2.imread(img_path)
    if img is None:
        return []
    h, w = img.shape[:2]
    
    label_path = os.path.join(labels_dir, l_file)
    try:
        with open(label_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except:
        return []
    
    records = []
    for idx, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue
        parts = line.split(',', 8)
        if len(parts) < 9:
            continue
        transcript = parts[8]
        if transcript == '###' or not transcript.strip():
            continue
        try:
            coords = [float(p) for p in parts[:8]]
        except ValueError:
            continue
        
        xs = [coords[0], coords[2], coords[4], coords[6]]
        ys = [coords[1], coords[3], coords[5], coords[7]]
        x1 = max(0, int(min(xs)))
        y1 = max(0, int(min(ys)))
        x2 = min(w, int(max(xs)))
        y2 = min(h, int(max(ys)))
        if (x2 - x1) <= 2 or (y2 - y1) <= 2:
            continue
        
        crop = img[y1:y2, x1:x2]
        crop_name = f'crop_{N:04d}_{idx:03d}.jpg'
        cv2.imwrite(os.path.join(crops_dir, crop_name), crop)
        records.append((is_train, f'crops/{crop_name}\t{transcript}\n'))
    return records

train_lines = []
val_lines = []

with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(process_label_file, label_files))

for res in results:
    for is_train, record in res:
        if is_train:
            train_lines.append(record)
        else:
            val_lines.append(record)

# Merge HuggingFace data into training set
if hf_labels:
    # 90/10 split for HF data
    split_idx = int(len(hf_labels) * 0.9)
    train_lines.extend(hf_labels[:split_idx])
    val_lines.extend(hf_labels[split_idx:])

print(f'VinText crops: {sum(1 for r in results for _ in r)}')
print(f'Total train: {len(train_lines)}, val: {len(val_lines)}')

In [ ]:
# Cell 5: Prepare train_data directory structure

train_data_dir = '/kaggle/working/train_data'
os.makedirs(train_data_dir, exist_ok=True)

# Create crops symlink inside train_data
crops_link = os.path.join(train_data_dir, 'crops')
if os.path.exists(crops_link):
    os.unlink(crops_link)
os.symlink(crops_dir, crops_link)

# Copy HF crops if they exist
hf_src = '/kaggle/working/hf_crops'
if os.path.exists(hf_src):
    hf_link = os.path.join(train_data_dir, 'hf_crops')
    if os.path.exists(hf_link):
        os.unlink(hf_link)
    os.symlink(hf_src, hf_link)

# Write label files
train_list = os.path.join(train_data_dir, 'train_list.txt')
val_list = os.path.join(train_data_dir, 'val_list.txt')

with open(train_list, 'w', encoding='utf-8') as f:
    f.writelines(train_lines)
with open(val_list, 'w', encoding='utf-8') as f:
    f.writelines(val_lines)

print(f'Train labels: {len(train_lines)}')
print(f'Val labels: {len(val_lines)}')
print('train_data directory ready.')

In [ ]:
# Cell 6: Download pretrained weights

import requests

pretrain_dir = '/kaggle/working/pretrain_models'
os.makedirs(pretrain_dir, exist_ok=True)

pretrained_path = os.path.join(pretrain_dir, 'latin_PP-OCRv5_mobile_rec_pretrained.pdparams')
if not os.path.exists(pretrained_path):
    url = 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/latin_PP-OCRv5_mobile_rec_pretrained.pdparams'
    print('Downloading pretrained weights...')
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(pretrained_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print('Done.')
else:
    print('Pretrained weights already exist.')

In [ ]:
# Cell 7: Fine-tune the model
# Uses PP-OCRv5-server Vietnamese config with:
# - Increased augmentations for diacritics
# - Higher resolution input (48x320)
# - Longer training with cosine LR schedule

!python tools/train.py \
    -c configs/rec/PP-OCRv5/multi_language/rec_vi_server.yml \
    -o Global.save_model_dir=/kaggle/working/output/vi_PP-OCRv5_server_rec \
    Global.pretrained_model=/kaggle/working/pretrain_models/latin_PP-OCRv5_mobile_rec_pretrained \
    Train.dataset.data_dir=/kaggle/working/train_data/ \
    Train.dataset.label_file_list=[/kaggle/working/train_data/train_list.txt] \
    Eval.dataset.data_dir=/kaggle/working/train_data/ \
    Eval.dataset.label_file_list=[/kaggle/working/train_data/val_list.txt] \
    Global.epoch_num=50 \
    Optimizer.lr.learning_rate=0.0005 \
    Train.loader.batch_size_per_card=64 \
    Train.loader.num_workers=4

In [ ]:
# Cell 8: Export model to inference format

!python tools/export_model.py \
    -c configs/rec/PP-OCRv5/multi_language/rec_vi_server.yml \
    -o Global.pretrained_model=/kaggle/working/output/vi_PP-OCRv5_server_rec/best_accuracy \
    Global.save_inference_dir=/kaggle/working/inference/vi_PP-OCRv5_server_rec

print('Model exported to inference format.')

In [ ]:
# Cell 9: Package model for upload

import zipfile

model_zip = '/kaggle/working/vi_PP-OCRv5_server_rec_inference.zip'
inference_dir = '/kaggle/working/inference/vi_PP-OCRv5_server_rec'

print('Packaging model...')
with zipfile.ZipFile(model_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(inference_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, '/kaggle/working/inference')
            zf.write(file_path, arcname)

print(f'Model packaged: {model_zip}')
print(f'Size: {os.path.getsize(model_zip) / 1024 / 1024:.1f} MB')
print('\nDownload this file and upload to Modal Volume:')
print('python -m modal volume put viet-ocr-vol vi_PP-OCRv5_server_rec_inference.zip inference/')')